# LeetCode 1394 — Find Lucky Integer in an Array bằng Python

**Link đề:** https://leetcode.com/problems/find-lucky-integer-in-an-array/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Counting

```python
class Solution:
    def findLucky(self, arr: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **tần suất** (frequency): một giá trị xuất hiện bao nhiêu lần.
2. Biết 3 công cụ đếm của Python: `list.count`, `dict` tự đếm, `collections.Counter`.
3. Đi từ cách trực giác `O(n²)` → đếm bằng hash map `O(n)` → mảng đếm theo giá trị (counting array).
4. Biết kỹ thuật "tìm **lớn nhất** thỏa điều kiện": giữ biến `best` hoặc duyệt từ lớn xuống nhỏ rồi dừng sớm.
5. Trace, chứng minh, Time/Space, lỗi hay gặp, bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho mảng số nguyên `arr`. Một số nguyên được gọi là **lucky** (may mắn) nếu **tần suất** của nó trong mảng **bằng chính giá trị** của nó.

Trả về **số lucky lớn nhất**. Nếu không có số lucky nào, trả về `-1`.

| `arr` | Output | Vì sao? |
|---|---:|---|
| `[2,2,3,4]` | `2` | `2` xuất hiện 2 lần; `3` xuất hiện 1 lần, `4` xuất hiện 1 lần |
| `[1,2,2,3,3,3]` | `3` | `1`, `2`, `3` đều lucky → lấy lớn nhất `3` |
| `[2,2,2,3,3]` | `-1` | `2` xuất hiện 3 lần, `3` xuất hiện 2 lần → không ai lucky |

**Constraints:** `1 <= arr.length <= 500`, `1 <= arr[i] <= 500`.

Câu hỏi trước khi code:
- "Tần suất của `x`" nghĩa là gì? Đếm thế nào?
- Nếu có nhiều số lucky thì lấy số nào?
- Có thể có số lucky mà **không** xuất hiện trong mảng không? (Gợi ý: tần suất của nó là 0, còn giá trị ≥ 1.)

In [1]:
examples = [
    ([2, 2, 3, 4], 2),
    ([1, 2, 2, 3, 3, 3], 3),
    ([2, 2, 2, 3, 3], -1),
]
for arr, expected in examples:
    print(f"arr = {arr} -> kỳ vọng {expected}")

arr = [2, 2, 3, 4] -> kỳ vọng 2
arr = [1, 2, 2, 3, 3, 3] -> kỳ vọng 3
arr = [2, 2, 2, 3, 3] -> kỳ vọng -1


## 2. Nền tảng Python cần biết

### 2.1. `list.count(x)` — đếm số lần `x` xuất hiện

`arr.count(x)` duyệt **toàn bộ** list và đếm số phần tử bằng `x`. Dễ dùng nhưng mỗi lần gọi tốn `O(n)`.

In [2]:
arr = [1, 2, 2, 3, 3, 3]
print("arr.count(3) =", arr.count(3))
print("arr.count(2) =", arr.count(2))
print("arr.count(7) =", arr.count(7))   # không có -> 0

arr.count(3) = 3
arr.count(2) = 2
arr.count(7) = 0


### 2.2. `dict` dùng làm bảng đếm (hash map)

`dict` lưu cặp **key → value**. Ở bài này: key = giá trị trong mảng, value = số lần xuất hiện.

- `freq.get(x, 0)` đọc value của `x`, nếu chưa có key thì trả `0` (không báo lỗi).
- `freq[x] = freq.get(x, 0) + 1` tăng bộ đếm của `x` lên 1.
- Tra cứu/cập nhật trung bình `O(1)`.

In [3]:
freq = {}
for x in [2, 2, 3, 4]:
    freq[x] = freq.get(x, 0) + 1
    print(f"đọc {x} -> freq = {freq}")

print("freq[2] =", freq[2])
print("freq.get(9, 0) =", freq.get(9, 0))

đọc 2 -> freq = {2: 1}
đọc 2 -> freq = {2: 2}
đọc 3 -> freq = {2: 2, 3: 1}
đọc 4 -> freq = {2: 2, 3: 1, 4: 1}
freq[2] = 2
freq.get(9, 0) = 0


### 2.3. `collections.Counter` — dict chuyên để đếm

`Counter(arr)` làm đúng việc của vòng lặp trên chỉ trong 1 dòng. Đặc biệt: truy cập key không tồn tại trả về `0` thay vì lỗi `KeyError`.

`.items()` trả về các cặp `(key, value)` để duyệt.

In [4]:
from collections import Counter

cnt = Counter([1, 2, 2, 3, 3, 3])
print(cnt)
print("cnt[3] =", cnt[3], "| cnt[100] =", cnt[100])
for value, times in cnt.items():
    print(f"giá trị {value} xuất hiện {times} lần -> lucky? {value == times}")

Counter({3: 3, 2: 2, 1: 1})
cnt[3] = 3 | cnt[100] = 0
giá trị 1 xuất hiện 1 lần -> lucky? True
giá trị 2 xuất hiện 2 lần -> lucky? True
giá trị 3 xuất hiện 3 lần -> lucky? True


### 2.4. `max(..., default=...)` và generator expression

- `max(iterable)` lấy phần tử lớn nhất. Nếu iterable **rỗng** sẽ lỗi `ValueError`.
- `max(iterable, default=-1)` trả `-1` khi rỗng → rất hợp với yêu cầu "không có thì trả -1".
- `(x for x in ... if điều_kiện)` là **generator expression**: sinh dần từng phần tử thỏa điều kiện, không tạo list trung gian.

In [5]:
print(max([3, 9, 4]))
print(max([], default=-1))
nums = [5, 8, 2, 7]
print(max((x for x in nums if x % 2 == 0), default=-1))   # số chẵn lớn nhất = 8
print(max((x for x in nums if x > 100), default=-1))      # không có -> -1

9
-1
8
-1


### 2.5. `range` đi lùi: `range(start, stop, -1)`

`range(5, 0, -1)` sinh `5, 4, 3, 2, 1` (dừng **trước** `0`). Duyệt từ lớn xuống nhỏ giúp **dừng sớm** ngay khi gặp số lucky đầu tiên — đó chắc chắn là số lớn nhất.

In [6]:
print(list(range(5, 0, -1)))
print(list(range(10, 6, -1)))

[5, 4, 3, 2, 1]
[10, 9, 8, 7]


## 3. Cách trực giác (baseline) — đếm lại cho từng phần tử

Người mới thường nghĩ: "Với mỗi `x` trong mảng, đếm xem `x` xuất hiện mấy lần bằng `arr.count(x)`. Nếu bằng `x` thì là lucky, giữ số lớn nhất."

```text
best = -1
với mỗi x trong arr:
    nếu arr.count(x) == x và x > best: best = x
trả best
```

In [7]:
def find_lucky_bruteforce(arr: list[int]) -> int:
    best = -1
    for x in arr:
        if arr.count(x) == x and x > best:
            best = x
    return best

for arr, expected in examples:
    print(arr, "->", find_lucky_bruteforce(arr), "| kỳ vọng", expected)

[2, 2, 3, 4] -> 2 | kỳ vọng 2
[1, 2, 2, 3, 3, 3] -> 3 | kỳ vọng 3
[2, 2, 2, 3, 3] -> -1 | kỳ vọng -1


### 3.1. Nút thắt của baseline

- Có `n` phần tử, mỗi phần tử gọi `arr.count` tốn `n` phép so sánh → khoảng `n²` phép.
- Với `n = 500` thì `250 000` phép — vẫn chạy được, nhưng **lặp lại công việc**: số `3` xuất hiện 3 lần thì ta đếm số `3` tới 3 lần!

Đo số phép so sánh để thấy rõ:

In [8]:
def count_comparisons_bruteforce(arr: list[int]) -> int:
    # mỗi arr.count(x) duyệt hết len(arr) phần tử
    return len(arr) * len(arr)

for n in [10, 100, 500]:
    arr = [3] * n
    print(f"n = {n:>3}: baseline ≈ {count_comparisons_bruteforce(arr):>7} phép so sánh, đếm 1 lần chỉ cần {n}")

n =  10: baseline ≈     100 phép so sánh, đếm 1 lần chỉ cần 10
n = 100: baseline ≈   10000 phép so sánh, đếm 1 lần chỉ cần 100
n = 500: baseline ≈  250000 phép so sánh, đếm 1 lần chỉ cần 500


## 4. Insight — đếm **một lần**, tra cứu nhiều lần

Câu hỏi "tần suất của x là bao nhiêu?" lặp đi lặp lại → **đếm trước một lần** vào bảng `freq`, sau đó mỗi câu hỏi chỉ là tra `freq[x]` trong `O(1)`.

Hai bước:
1. **Đếm:** duyệt mảng một lần, xây `freq`.
2. **Lọc + chọn:** duyệt các cặp `(value, times)` trong `freq`; nếu `value == times` thì là ứng viên; giữ ứng viên lớn nhất.

Lưu ý: ở bước 2 ta duyệt các **giá trị khác nhau** (key), không duyệt lại từng phần tử của mảng.

In [9]:
arr = [1, 2, 2, 3, 3, 3]
freq = Counter(arr)
candidates = [value for value, times in freq.items() if value == times]
print("freq       =", dict(freq))
print("candidates =", candidates)
print("đáp án     =", max(candidates) if candidates else -1)

freq       = {1: 1, 2: 2, 3: 3}
candidates = [1, 2, 3]
đáp án     = 3


## 5. Lời giải chính — hash map, giảng từng đoạn

**Đoạn 1 — đếm:** `freq = {}` rồi tăng `freq[x]` cho từng `x`.

**Đoạn 2 — khởi tạo đáp án:** `best = -1` (đúng giá trị phải trả nếu không có số lucky).

**Đoạn 3 — duyệt các cặp:** `for value, times in freq.items()`; nếu `value == times` và `value > best` thì cập nhật `best`.

**Đoạn 4 — trả `best`.**

In [10]:
def find_lucky_hashmap(arr: list[int]) -> int:
    # Đoạn 1: đếm tần suất
    freq = {}
    for x in arr:
        freq[x] = freq.get(x, 0) + 1
    # Đoạn 2: mặc định "không có"
    best = -1
    # Đoạn 3: lọc số lucky, giữ số lớn nhất
    for value, times in freq.items():
        if value == times and value > best:
            best = value
    # Đoạn 4
    return best

for arr, expected in examples:
    print(arr, "->", find_lucky_hashmap(arr), "| kỳ vọng", expected)

[2, 2, 3, 4] -> 2 | kỳ vọng 2
[1, 2, 2, 3, 3, 3] -> 3 | kỳ vọng 3
[2, 2, 2, 3, 3] -> -1 | kỳ vọng -1


### 5.1. Bản gọn với `Counter` + `max(default=-1)`

Cùng ý tưởng, viết ngắn bằng thư viện chuẩn:

In [11]:
def find_lucky_counter(arr: list[int]) -> int:
    cnt = Counter(arr)
    return max((v for v, t in cnt.items() if v == t), default=-1)

for arr, expected in examples:
    print(arr, "->", find_lucky_counter(arr), "| kỳ vọng", expected)

[2, 2, 3, 4] -> 2 | kỳ vọng 2
[1, 2, 2, 3, 3, 3] -> 3 | kỳ vọng 3
[2, 2, 2, 3, 3] -> -1 | kỳ vọng -1


## 6. Cách 3 — mảng đếm (counting array) + duyệt từ lớn xuống

Vì `1 <= arr[i] <= 500`, ta có thể dùng **list** độ dài 501 thay cho dict: `count[v]` = số lần `v` xuất hiện. Chỉ số của list chính là giá trị.

Sau đó duyệt `v` từ `500` xuống `1`: số đầu tiên có `count[v] == v` chính là **lucky lớn nhất** → trả luôn, không cần duyệt tiếp.

Ý tưởng "dùng giá trị làm chỉ số" gọi là **counting sort / bucket** — chỉ dùng được khi miền giá trị nhỏ và biết trước.

In [12]:
def find_lucky_array(arr: list[int]) -> int:
    count = [0] * 501              # count[v] cho v = 0..500
    for x in arr:
        count[x] += 1
    for v in range(500, 0, -1):     # từ lớn xuống: gặp là lớn nhất
        if count[v] == v:
            return v
    return -1

for arr, expected in examples:
    print(arr, "->", find_lucky_array(arr), "| kỳ vọng", expected)

[2, 2, 3, 4] -> 2 | kỳ vọng 2
[1, 2, 2, 3, 3, 3] -> 3 | kỳ vọng 3
[2, 2, 2, 3, 3] -> -1 | kỳ vọng -1


### 6.1. Một nhận xét giúp thu hẹp: số lucky luôn `<= len(arr)`

Nếu `v` lucky thì `v` xuất hiện `v` lần, nên `v <= n`. Vì vậy chỉ cần duyệt `v` từ `n` xuống `1` — các giá trị lớn hơn `n` không bao giờ lucky.

In [13]:
arr = [7, 7, 7, 7, 7, 7, 7, 1]
n = len(arr)
print("n =", n, "| các giá trị có thể lucky: 1 ..", n)
print("find_lucky_array:", find_lucky_array(arr))

n = 8 | các giá trị có thể lucky: 1 .. 8
find_lucky_array: 7


## 7. Cách 4 — sắp xếp giảm dần rồi đếm theo nhóm

Sắp xếp `arr` **giảm dần**: các số bằng nhau đứng liền nhau thành **nhóm**, và nhóm lớn đứng trước. Duyệt từng nhóm, đo độ dài nhóm; nhóm đầu tiên có `độ dài == giá trị` là đáp án.

Đây là cách **không dùng hash map** — hay gặp khi ngôn ngữ/đề cấm cấu trúc phụ. Chi phí do sắp xếp: `O(n log n)`.

In [14]:
def find_lucky_sort(arr: list[int]) -> int:
    a = sorted(arr, reverse=True)
    i = 0
    while i < len(a):
        j = i
        while j < len(a) and a[j] == a[i]:   # mở rộng nhóm cùng giá trị
            j += 1
        if j - i == a[i]:                     # độ dài nhóm == giá trị
            return a[i]
        i = j                                 # nhảy sang nhóm kế
    return -1

for arr, expected in examples:
    print(arr, "->", find_lucky_sort(arr), "| kỳ vọng", expected)

[2, 2, 3, 4] -> 2 | kỳ vọng 2
[1, 2, 2, 3, 3, 3] -> 3 | kỳ vọng 3
[2, 2, 2, 3, 3] -> -1 | kỳ vọng -1


`sorted(arr, reverse=True)` trả về **list mới** đã sắp giảm dần, **không** sửa `arr` gốc (khác `arr.sort()` sửa tại chỗ).

In [15]:
arr = [2, 3, 2, 4]
print("sorted giảm dần:", sorted(arr, reverse=True))
print("arr gốc vẫn là :", arr)

sorted giảm dần: [4, 3, 2, 2]
arr gốc vẫn là : [2, 3, 2, 4]


## 8. TRACE từng bước

Trace lời giải hash map với `arr = [1, 2, 2, 3, 3, 3]`.

**Pha 1 — đếm:**

| i | x | freq sau khi cập nhật |
|---:|---:|---|
| 0 | 1 | `{1:1}` |
| 1 | 2 | `{1:1, 2:1}` |
| 2 | 2 | `{1:1, 2:2}` |
| 3 | 3 | `{1:1, 2:2, 3:1}` |
| 4 | 3 | `{1:1, 2:2, 3:2}` |
| 5 | 3 | `{1:1, 2:2, 3:3}` |

**Pha 2 — chọn:**

| value | times | value == times? | best trước | best sau |
|---:|---:|:---:|---:|---:|
| 1 | 1 | ✔ | -1 | 1 |
| 2 | 2 | ✔ | 1 | 2 |
| 3 | 3 | ✔ | 2 | 3 |

Kết quả `3`.

In [16]:
def trace_find_lucky(arr: list[int]) -> int:
    print(f"arr = {arr}")
    print("Pha 1: đếm")
    freq = {}
    for i, x in enumerate(arr):
        freq[x] = freq.get(x, 0) + 1
        print(f"  i={i}, x={x} -> freq={freq}")
    print("Pha 2: chọn số lucky lớn nhất")
    best = -1
    for value, times in freq.items():
        ok = value == times
        old = best
        if ok and value > best:
            best = value
        print(f"  value={value}, times={times}, lucky={ok!s:<5} best: {old} -> {best}")
    print("=> trả về", best)
    return best

trace_find_lucky([1, 2, 2, 3, 3, 3])
print()
trace_find_lucky([2, 2, 2, 3, 3])

arr = [1, 2, 2, 3, 3, 3]
Pha 1: đếm
  i=0, x=1 -> freq={1: 1}
  i=1, x=2 -> freq={1: 1, 2: 1}
  i=2, x=2 -> freq={1: 1, 2: 2}
  i=3, x=3 -> freq={1: 1, 2: 2, 3: 1}
  i=4, x=3 -> freq={1: 1, 2: 2, 3: 2}
  i=5, x=3 -> freq={1: 1, 2: 2, 3: 3}
Pha 2: chọn số lucky lớn nhất
  value=1, times=1, lucky=True  best: -1 -> 1
  value=2, times=2, lucky=True  best: 1 -> 2
  value=3, times=3, lucky=True  best: 2 -> 3
=> trả về 3

arr = [2, 2, 2, 3, 3]
Pha 1: đếm
  i=0, x=2 -> freq={2: 1}
  i=1, x=2 -> freq={2: 2}
  i=2, x=2 -> freq={2: 3}
  i=3, x=3 -> freq={2: 3, 3: 1}
  i=4, x=3 -> freq={2: 3, 3: 2}
Pha 2: chọn số lucky lớn nhất
  value=2, times=3, lucky=False best: -1 -> -1
  value=3, times=2, lucky=False best: -1 -> -1
=> trả về -1


-1

`enumerate(arr)` sinh cặp `(chỉ_số, giá_trị)` để vừa có `i` vừa có `x`. Trong f-string, `{ok!s:<5}` đổi `ok` sang chuỗi rồi căn trái độ rộng 5 cho bảng thẳng hàng.

Ví dụ thứ hai cho thấy nhánh **không có ai lucky**: `best` giữ nguyên `-1` suốt pha 2.

Trace thêm cách mảng đếm với ví dụ có giá trị lớn không lucky để thấy **dừng sớm**:

In [17]:
def trace_find_lucky_array(arr: list[int]) -> int:
    count = [0] * (len(arr) + 1)
    for x in arr:
        if x <= len(arr):            # giá trị > n không thể lucky, bỏ qua
            count[x] += 1
    print("count (chỉ số 0..n):", count)
    for v in range(len(arr), 0, -1):
        print(f"  v={v}: count[v]={count[v]} -> {'DỪNG, lucky!' if count[v] == v else 'không'}")
        if count[v] == v:
            return v
    return -1

print("kết quả:", trace_find_lucky_array([4, 4, 4, 4, 3, 3, 2, 2, 9]))

count (chỉ số 0..n): [0, 0, 2, 2, 4, 0, 0, 0, 0, 1]
  v=9: count[v]=1 -> không
  v=8: count[v]=0 -> không
  v=7: count[v]=0 -> không
  v=6: count[v]=0 -> không
  v=5: count[v]=0 -> không
  v=4: count[v]=4 -> DỪNG, lucky!
kết quả: 4


## 9. Code nộp LeetCode

### Bản hash map (khuyên dùng)

In [18]:
from collections import Counter

class Solution:
    def findLucky(self, arr: list[int]) -> int:
        cnt = Counter(arr)
        best = -1
        for value, times in cnt.items():
            if value == times and value > best:
                best = value
        return best

print([Solution().findLucky(a) for a, _ in examples])

[2, 3, -1]


### Bản mảng đếm, dừng sớm

In [19]:
class SolutionArray:
    def findLucky(self, arr: list[int]) -> int:
        n = len(arr)
        count = [0] * (n + 1)
        for x in arr:
            if x <= n:
                count[x] += 1
        for v in range(n, 0, -1):
            if count[v] == v:
                return v
        return -1

print([SolutionArray().findLucky(a) for a, _ in examples])

[2, 3, -1]


Trên LeetCode, `List[int]` (từ `typing`) và `list[int]` đều được chấp nhận với Python 3.9+. Notebook dùng `list[int]` để khỏi phải `import`.

### Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Baseline `arr.count` | đếm lại cho từng phần tử | `O(n²)` | `O(1)` | dễ nghĩ, lặp việc |
| 2. Hash map / `Counter` | đếm 1 lần, lọc theo key | `O(n)` trung bình | `O(k)` (k = số giá trị khác nhau) | tổng quát nhất |
| 3. Mảng đếm + duyệt lùi | giá trị làm chỉ số, dừng sớm | `O(n)` | `O(n)` | cần miền giá trị nhỏ |
| 4. Sắp xếp giảm + nhóm | nhóm liền nhau | `O(n log n)` | `O(n)` (bản sao đã sắp) | không cần hash |

## 10. Vì sao đúng?

**Bất biến pha 1:** sau khi xử lý `i` phần tử đầu, `freq[v]` = số lần `v` xuất hiện trong `arr[0..i-1]`. Hết vòng lặp ⇒ `freq[v]` đúng bằng tần suất trong cả mảng.

**Bất biến pha 2:** sau khi duyệt một số cặp, `best` = số lucky lớn nhất **trong các cặp đã duyệt**, hoặc `-1` nếu chưa có. Mọi số lucky đều là key của `freq` (vì nó xuất hiện ≥ 1 lần) ⇒ duyệt hết các key là không bỏ sót.

**Không trả nhầm:** `best` chỉ được gán bằng `value` khi `value == times`, tức là đúng định nghĩa lucky.

**Cách mảng đếm:** duyệt `v` giảm dần; lần đầu gặp `count[v] == v` thì mọi `v' > v` đã bị loại ⇒ `v` lớn nhất. Giá trị `> n` bị bỏ qua là an toàn vì không thể lucky.

## 11. Time / Space

Gọi `n = len(arr)`, `k` = số giá trị khác nhau (`k <= n`).

- **Hash map:** đếm `O(n)` trung bình (mỗi thao tác dict trung bình `O(1)`), duyệt key `O(k)` ⇒ `O(n)`. Bộ nhớ `O(k)`.
- **Mảng đếm:** `O(n)` đếm + `O(n)` duyệt lùi ⇒ `O(n)`; bộ nhớ `O(n)` (hoặc `O(501)` = hằng số nếu cấp phát theo constraint).
- **Baseline:** `O(n²)` thời gian, `O(1)` bộ nhớ.
- **Sắp xếp:** `O(n log n)` thời gian.

In [20]:
import time, random

random.seed(1)
big = [random.randint(1, 500) for _ in range(500)]
for f in [find_lucky_bruteforce, find_lucky_hashmap, find_lucky_array, find_lucky_sort]:
    t0 = time.perf_counter()
    for _ in range(50):
        f(big)
    print(f"{f.__name__:<24} {1000 * (time.perf_counter() - t0):7.2f} ms cho 50 lần")

find_lucky_bruteforce     108.14 ms cho 50 lần
find_lucky_hashmap          2.87 ms cho 50 lần


find_lucky_array            2.77 ms cho 50 lần
find_lucky_sort             6.41 ms cho 50 lần


(Thời gian đo thay đổi theo máy; điều cần thấy là baseline chậm hơn rõ rệt.)

## 12. Bộ test

Nhóm test: ví dụ đề, mảng 1 phần tử (`[1]` lucky, `[2]` không), nhiều số lucky, số lucky không phải giá trị lớn nhất trong mảng, giá trị lớn nhất 500, và **đối chiếu ngẫu nhiên** mọi cách với baseline.

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<34} PASS -> {actual}")

sol = Solution()
check("ví dụ 1", sol.findLucky([2, 2, 3, 4]), 2)
check("ví dụ 2", sol.findLucky([1, 2, 2, 3, 3, 3]), 3)
check("ví dụ 3", sol.findLucky([2, 2, 2, 3, 3]), -1)
check("một phần tử [1]", sol.findLucky([1]), 1)
check("một phần tử [2]", sol.findLucky([2]), -1)
check("lucky nhỏ, có số lớn hơn không lucky", sol.findLucky([5, 1, 7, 7]), 1)
check("toàn 500 (n=500)", sol.findLucky([500] * 500), 500)
check("499 lần 500 -> không", sol.findLucky([500] * 499 + [1]), 1)
check("thứ tự lộn xộn", sol.findLucky([3, 1, 3, 2, 3, 2]), 3)

ví dụ 1                            PASS -> 2
ví dụ 2                            PASS -> 3
ví dụ 3                            PASS -> -1
một phần tử [1]                    PASS -> 1
một phần tử [2]                    PASS -> -1
lucky nhỏ, có số lớn hơn không lucky PASS -> 1
toàn 500 (n=500)                   PASS -> 500
499 lần 500 -> không               PASS -> 1
thứ tự lộn xộn                     PASS -> 3


In [22]:
methods = [find_lucky_hashmap, find_lucky_counter, find_lucky_array, find_lucky_sort,
           Solution().findLucky, SolutionArray().findLucky]
random.seed(1394)
for _ in range(3000):
    n = random.randint(1, 30)
    arr = [random.randint(1, 8) for _ in range(n)]
    expected = find_lucky_bruteforce(arr)
    for f in methods:
        assert f(arr) == expected, (f, arr)
print("Mọi cách khớp baseline trên 3000 mảng ngẫu nhiên")

Mọi cách khớp baseline trên 3000 mảng ngẫu nhiên


## 13. Lỗi người mới hay gặp

### Lỗi 1: Trả số lucky **đầu tiên** gặp thay vì lớn nhất

Sai: duyệt `freq` theo thứ tự xuất hiện và `return` ngay. Dict giữ thứ tự **chèn vào**, không phải thứ tự lớn → nhỏ.

In [23]:
def wrong_first_lucky(arr):
    for value, times in Counter(arr).items():
        if value == times:
            return value          # SAI: chưa chắc lớn nhất
    return -1

arr = [1, 2, 2]
print("sai:", wrong_first_lucky(arr), "| đúng:", find_lucky_hashmap(arr))

sai: 1 | đúng: 2


**Sửa:** giữ `best` và so sánh, hoặc duyệt giá trị theo thứ tự giảm dần (mảng đếm / `sorted(..., reverse=True)`).

### Lỗi 2: So sánh tần suất với **chỉ số** thay vì giá trị

In [24]:
def wrong_index(arr):
    best = -1
    for i, x in enumerate(arr):
        if arr.count(x) == i:        # SAI: dùng i (vị trí) thay vì x (giá trị)
            best = max(best, x)
    return best

print("sai:", wrong_index([2, 2, 3, 4]), "| đúng:", find_lucky_hashmap([2, 2, 3, 4]))

sai: -1 | đúng: 2


### Lỗi 3: `max(...)` trên danh sách rỗng

`max([])` báo `ValueError`. Khi không có ứng viên phải trả `-1`.

In [25]:
try:
    max(v for v, t in Counter([2, 2, 2]).items() if v == t)
except ValueError as e:
    print("Lỗi:", e)
print("Sửa bằng default:", max((v for v, t in Counter([2, 2, 2]).items() if v == t), default=-1))

Lỗi: max() iterable argument is empty
Sửa bằng default: -1


### Lỗi 4: Mảng đếm cấp phát thiếu

`count = [0] * 500` chỉ có chỉ số `0..499` → `count[500]` báo `IndexError`. Cần `[0] * 501`, hoặc bỏ qua giá trị `> n` như `SolutionArray`.

In [26]:
try:
    count = [0] * 500
    count[500] += 1
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


## 14. Mẹo nhận dạng pattern

- Đề nói "**tần suất / số lần xuất hiện / frequency**" ⇒ nghĩ ngay `Counter` / dict đếm.
- "Lớn nhất/nhỏ nhất **thỏa điều kiện**, không có thì -1" ⇒ biến `best = -1` + cập nhật, hoặc `max(..., default=-1)`.
- Miền giá trị nhỏ (≤ vài nghìn) ⇒ có thể thay dict bằng **mảng đếm**, duyệt theo thứ tự giá trị.
- Quan sát giới hạn tự nhiên (`v <= n`) giúp bỏ bớt ứng viên.

## 15. Biến thể

1. **Trả về tất cả số lucky** (tăng dần).
2. **Lucky nhỏ nhất** thay vì lớn nhất.
3. **"Gần lucky"**: tần suất bằng giá trị ± 1.
4. LeetCode 1207 *Unique Number of Occurrences*: kiểm tra các tần suất có đôi một khác nhau — cũng dùng `Counter`.

In [27]:
def all_lucky(arr: list[int]) -> list[int]:
    cnt = Counter(arr)
    return sorted(v for v, t in cnt.items() if v == t)

def smallest_lucky(arr: list[int]) -> int:
    return min((v for v, t in Counter(arr).items() if v == t), default=-1)

def unique_occurrences(arr: list[int]) -> bool:      # LeetCode 1207
    times = list(Counter(arr).values())
    return len(times) == len(set(times))

print(all_lucky([1, 2, 2, 3, 3, 3, 4]))
print(smallest_lucky([2, 2, 3, 3, 3]))
print(unique_occurrences([1, 2, 2, 1, 1, 3]), unique_occurrences([1, 2]))

[1, 2, 3]
2
True False


## 16. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** Với `arr = [4, 4, 4, 4, 1, 3, 3]`, viết bảng `freq` và cho biết đáp án.

**Bài 2 (đoán output):** `Counter([5, 5, 1])[5]`, `Counter([5, 5, 1])[9]`, `max([], default=-1)`, `list(range(3, 0, -1))`.

**Bài 3 (điền chỗ trống):** Hoàn thiện `exercise_find_lucky(arr)` dùng dict tự đếm (không dùng `Counter`).

**Bài 4 (tự viết):** `exercise_lucky_desc(arr)` dùng mảng đếm và duyệt từ lớn xuống, dừng sớm.

**Bài 5 (biến thể):** `exercise_all_lucky(arr)` trả về list mọi số lucky theo thứ tự **giảm dần**.

**Bài 6 (mini challenge):** `exercise_min_removals(arr)` — số phần tử **ít nhất** cần xóa khỏi `arr` để mảng có **ít nhất một** số lucky (nếu đã có thì `0`). Gợi ý: với mỗi giá trị `v` xuất hiện `t >= v` lần, xóa `t - v` bản sao.

In [28]:
def exercise_find_lucky(arr: list[int]) -> int:
    # TODO 1: tạo dict freq rỗng
    # TODO 2: đếm tần suất bằng freq.get(x, 0) + 1
    # TODO 3: best = -1, duyệt freq.items(), cập nhật best
    pass

def exercise_lucky_desc(arr: list[int]) -> int:
    # TODO: mảng đếm + range(n, 0, -1)
    pass

# assert exercise_find_lucky([2, 2, 3, 4]) == 2
# assert exercise_lucky_desc([1, 2, 2, 3, 3, 3]) == 3

In [29]:
def exercise_all_lucky(arr: list[int]) -> list[int]:
    # TODO
    pass

def exercise_min_removals(arr: list[int]) -> int:
    # TODO
    pass

# assert exercise_all_lucky([1, 2, 2, 3, 3, 3]) == [3, 2, 1]
# assert exercise_min_removals([2, 2, 2, 3, 3]) == 1

## 17. Gợi ý 3 mức

**Bài 1** — (1) Đếm từng giá trị. (2) `4` xuất hiện 4 lần, `1` một lần, `3` hai lần. (3) Lucky: `4` và `1` → lấy lớn nhất.

**Bài 3** — (1) Hai pha: đếm rồi chọn. (2) `freq[x] = freq.get(x, 0) + 1`. (3) `if value == times and value > best: best = value`.

**Bài 4** — (1) `count = [0] * (n + 1)`. (2) Bỏ qua `x > n`. (3) `for v in range(n, 0, -1): if count[v] == v: return v`.

**Bài 5** — (1) Lọc như bài chính. (2) Gom vào list. (3) `sorted(..., reverse=True)`.

**Bài 6** — (1) Nếu đã có lucky → 0. (2) Giá trị `v` chỉ "cứu" được khi `t >= v` (xóa bớt được, không thêm được). (3) Đáp án = `min(t - v)` trên các `v` có `t >= v`; không có thì không làm được (giả sử trả `-1`).

## 18. Đáp án tham khảo

**Bài 1:** `freq = {4: 4, 1: 1, 3: 2}` → lucky `{4, 1}` → **4**.

**Bài 2:** `2`, `0`, `-1`, `[3, 2, 1]`.

In [30]:
def answer_find_lucky(arr):
    freq = {}
    for x in arr:
        freq[x] = freq.get(x, 0) + 1
    best = -1
    for value, times in freq.items():
        if value == times and value > best:
            best = value
    return best

def answer_lucky_desc(arr):
    n = len(arr)
    count = [0] * (n + 1)
    for x in arr:
        if x <= n:
            count[x] += 1
    for v in range(n, 0, -1):
        if count[v] == v:
            return v
    return -1

def answer_all_lucky(arr):
    return sorted((v for v, t in Counter(arr).items() if v == t), reverse=True)

def answer_min_removals(arr):
    costs = [t - v for v, t in Counter(arr).items() if t >= v]
    return min(costs) if costs else -1

assert answer_find_lucky([4, 4, 4, 4, 1, 3, 3]) == 4
assert Counter([5, 5, 1])[5] == 2 and Counter([5, 5, 1])[9] == 0
assert max([], default=-1) == -1 and list(range(3, 0, -1)) == [3, 2, 1]
assert answer_find_lucky([2, 2, 3, 4]) == 2
assert answer_lucky_desc([1, 2, 2, 3, 3, 3]) == 3
assert answer_all_lucky([1, 2, 2, 3, 3, 3]) == [3, 2, 1]
assert answer_min_removals([2, 2, 2, 3, 3]) == 1
assert answer_min_removals([2, 3, 3]) == -1
assert answer_min_removals([1, 5]) == 0
random.seed(7)
for _ in range(1000):
    arr = [random.randint(1, 6) for _ in range(random.randint(1, 15))]
    assert answer_find_lucky(arr) == answer_lucky_desc(arr) == find_lucky_bruteforce(arr)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 19. Template nhớ nhanh + checklist submit

```python
from collections import Counter

class Solution:
    def findLucky(self, arr: list[int]) -> int:
        cnt = Counter(arr)                       # 1. đếm một lần
        return max((v for v, t in cnt.items()    # 2. lọc v == tần suất
                    if v == t), default=-1)      # 3. lớn nhất, rỗng -> -1
```

**Khung tư duy chung "đếm rồi lọc":** đếm tần suất → lọc theo điều kiện trên `(giá_trị, số_lần)` → chọn min/max/đếm.

**Checklist:**
- [ ] So sánh **giá trị** với **tần suất**, không phải với chỉ số.
- [ ] Lấy số lucky **lớn nhất**, không phải số gặp đầu tiên.
- [ ] Không có số lucky → trả `-1` (dùng `default=-1` hoặc `best = -1`).
- [ ] Nếu dùng mảng đếm: kích thước đủ (`501` hoặc bỏ qua `x > n`).